# LAMPOSE — train Kavya's voice (Piper)

Run the cells **one at a time, top to bottom**. Click the ▶ button on a cell,
wait for the green tick, then do the next one.

If a cell shows a red error, stop and send the error text to Claude.
Do not skip ahead — each cell needs the one before it.


## Step 1 — check we have a GPU

First set the runtime: menu **Runtime → Change runtime type → T4 GPU → Save**.
Then run this cell. It must print a table with `Tesla T4` in it.


In [ ]:
!nvidia-smi


## Step 2 — install Piper's training code

Takes about 3–5 minutes. Some red warning text is normal;
only an actual **Error** matters.


In [ ]:
!git clone -q https://github.com/rhasspy/piper.git /content/piper
%cd /content/piper/src/python
!pip install -q --upgrade pip wheel setuptools
!pip install -q -e .
!pip install -q 'pytorch-lightning~=1.9' 'torchmetrics<1.0' onnx onnxruntime
!bash build_monotonic_align.sh
print('\ninstall finished')


## Step 3 — upload the dataset

Run this cell, click **Choose Files**, and pick `piper_dataset.zip`
from your computer. It is about 40 MB, so it takes a minute.

*(If the upload keeps failing, put the zip in Google Drive instead and use
the Drive cell further down.)*


In [ ]:
from google.colab import files
up = files.upload()          # choose piper_dataset.zip
print(list(up))


In [ ]:
# unzip it — you should see 408 wav files
!rm -rf /content/dataset && mkdir -p /content/dataset
!unzip -q -o piper_dataset.zip -d /content/tmp
!mv /content/tmp/piper/* /content/dataset/
!ls /content/dataset
!ls /content/dataset/wavs | wc -l
!head -3 /content/dataset/metadata.csv


### Alternative: load from Google Drive instead
Only use this if the upload above did not work.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !rm -rf /content/dataset && mkdir -p /content/dataset /content/tmp
# !unzip -q -o '/content/drive/MyDrive/piper_dataset.zip' -d /content/tmp
# !mv /content/tmp/piper/* /content/dataset/
# !ls /content/dataset/wavs | wc -l


## Step 4 — prepare the data for training

This turns the Telugu text into phonemes using espeak.
It should end by printing the number of utterances.


In [ ]:
%cd /content/piper/src/python
!python3 -m piper_train.preprocess \
  --language te \
  --input-dir /content/dataset \
  --output-dir /content/training \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 16000
!ls -la /content/training | head
!wc -l /content/training/dataset.jsonl


## Step 5 — download the starting model

We do not train from nothing — we start from an English voice and teach it
her voice instead. This is why 24 minutes of audio is enough.

The file is 846 MB, so this takes a few minutes.


In [ ]:
!wget -q --show-progress -O /content/base.ckpt \
  'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/low/epoch%3D2307-step%3D558536.ckpt'
!ls -lh /content/base.ckpt


## Step 6 — train

**This is the long one.** Leave it running for 1–2 hours for the practice run.

You will see lines with `Epoch 0:` counting up. That means it is working.

**To stop it:** press the ■ stop button on the cell. That is normal and safe —
it saves as it goes, so whatever it has learned so far is kept.

Keep this browser tab open, or Colab will disconnect you.


In [ ]:
%cd /content/piper/src/python
!python3 -m piper_train \
  --dataset-dir /content/training \
  --accelerator gpu --devices 1 \
  --batch-size 16 \
  --validation-split 0.0 --num-test-examples 0 \
  --max_epochs 4000 \
  --checkpoint-epochs 5 \
  --precision 32 \
  --resume_from_checkpoint /content/base.ckpt


## Step 7 — turn it into a voice file

Finds the newest saved checkpoint and converts it to the `.onnx` file
the phone agent can use.


In [ ]:
import glob, os, shutil
cks = sorted(glob.glob('/content/training/lightning_logs/*/checkpoints/*.ckpt'),
             key=os.path.getmtime)
print('checkpoints found:', len(cks))
last = cks[-1]
print('using:', last)
!python3 -m piper_train.export_onnx '{last}' /content/kavya.onnx
shutil.copy('/content/training/config.json', '/content/kavya.onnx.json')
!ls -lh /content/kavya.onnx*


## Step 8 — hear it before you download

A quick listen. If this sounds like her — even roughly — the toolchain works.


In [ ]:
!pip -q install piper-tts
text = 'హలో నమస్తే సర్! మీరు సుమా ఓనర్ గారేనా?'
open('/content/say.txt','w').write(text)
!cat /content/say.txt | piper --model /content/kavya.onnx --output_file /content/test.wav
from IPython.display import Audio, display
display(Audio('/content/test.wav'))


## Step 9 — download the voice

Downloads two files. Send **both** to Claude.


In [ ]:
from google.colab import files
files.download('/content/kavya.onnx')
files.download('/content/kavya.onnx.json')
